# Stage 4 — Model training and OOF prediction

This notebook is the single Stage-4 orchestration entry point. The reusable implementations live in `src/models/`; the notebook loads protected artifacts, runs each model, and displays aggregate diagnostics.

> **PhysioNet DUA:** run only in the controlled Kaggle/Colab environment containing the protected Stage-2/3 artifacts. Never display patient rows or identifiers. OOF predictions and fitted models must remain in the gitignored `data/` and `results/` directories.

## Stage-4 protocol

1. Reuse the frozen 20% internal holdout and five development folds from Stage 3.
2. Fit imputation and numeric scaling on original training-fold rows, then run SMOTENC, then one-hot encode categories. Never refit a scaler on synthetic rows. Class weights are also training-fold only.
3. Select hyperparameters by mean development-fold AUROC; use mean AUPRC as the tie-breaker.
4. Produce exactly one out-of-fold probability for every development row.
5. Refit the selected configuration on the complete development set.
6. Do not fit, predict, evaluate, or tune against the internal test partition in this stage.
7. Leave F1-threshold selection and final model comparison to Stage 5.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_PROCESSED, N_CV_FOLDS, RANDOM_SEED
from src.models.classic import (
    logistic_regression_candidates,
    load_static_stage4_inputs,
    mlp_candidates,
    save_static_training_result,
    svm_candidates,
    train_logistic_regression,
    train_mlp,
    train_svm,
    train_xgboost,
    xgboost_candidates,
)

# Change the suffix when starting a different experiment.
ARTIFACT_SUFFIX = 'simple_v1'

np.random.seed(RANDOM_SEED)
print(f'Project root: {ROOT}')
print(f'Frozen CV folds: {N_CV_FOLDS}; random seed: {RANDOM_SEED}')

## 1. Load and revalidate protected Stage-2/3 artifacts

The loader checks row alignment, frozen assignments, all five folds, and patient separation. This cell prints aggregate counts only.

In [ ]:
static, splits = load_static_stage4_inputs()
assignments = splits.assignments
dev = assignments['split'].eq('dev')

print(f'Total rows: {len(static):,}')
print(f'Development rows: {len(splits.dev_indices):,}')
print(f'Internal-test rows (sealed): {len(splits.test_indices):,}')
print(f'Development prevalence: {assignments.loc[dev, "label"].mean():.3%}')

### Preprocessing

Fit numeric imputation/scaling on original training rows, apply SMOTENC, then one-hot encode categories. Use the existing Stage-2 features and frozen Stage-3 splits.

## 2. Inspect the imbalance-screening space

The smoke profile has three configurations and validates baseline, fold-derived class weighting, and SMOTENC on all five folds. The screening profile fixes LR at `C=1`, L2 and compares six mutually exclusive imbalance strategies, isolating the effect of imbalance handling. SMOTENC targets minority/majority ratios of 0.10, 0.25, 0.50, or 1.00 and runs before one-hot encoding; it is never combined with class weighting. Full LR hyperparameter tuning is intentionally deferred until the strategy shortlist is fixed.

In [ ]:
lr_smoke_candidates = logistic_regression_candidates(profile='smoke')
lr_screening_candidates = logistic_regression_candidates(profile='screening')
print(f'LR Smoke candidates: {len(lr_smoke_candidates)}')
print(f'LR Imbalance-screening candidates: {len(lr_screening_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in lr_screening_candidates]))

In [ ]:
xgb_smoke_candidates = xgboost_candidates(profile="smoke")
xgb_screening_candidates = xgboost_candidates(profile="screening")
print(f'XGBoost Smoke candidates: {len(xgb_smoke_candidates)}')
print(f'XGBoost Imbalance-screening candidates: {len(xgb_screening_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in xgb_screening_candidates]))

## 3. Smoke-run

This still uses all five frozen development folds, comparing baseline, fold-derived class weighting, and SMOTENC 0.25 for each model. LR uses `C=1`, L2. Smoke artifacts receive the version suffix plus `_smoke` and cannot overwrite the screening run.

In [ ]:
lr_smoke_result = train_logistic_regression(
    static,
    splits,
    profile='smoke',
    progress_callback=print,
)
lr_smoke_artifacts = save_static_training_result(
    lr_smoke_result, artifact_suffix=f'{ARTIFACT_SUFFIX}_smoke'
)
display(lr_smoke_result.strategy_metrics)
print(f'LR smoke best candidate: {lr_smoke_result.best_candidate.name}')

In [ ]:
xgb_smoke_result = train_xgboost(
    static,
    splits,
    profile="smoke",
    device="cuda",
    progress_callback=print,
)
xgb_smoke_artifacts = save_static_training_result(
    xgb_smoke_result, artifact_suffix=f'{ARTIFACT_SUFFIX}_smoke'
)
display(xgb_smoke_result.strategy_metrics)
print(f'XGBoost smoke best candidate: {xgb_smoke_result.best_candidate.name}')

## 4. Run the imbalance screening

Run this after checking the smoke results. Change each model's gate to `True`; each screening performs 6 × 5 fitted pipelines across six strategies, then refits its winner on dev. Artifacts use `ARTIFACT_SUFFIX`, preserving the previous pipeline's results.

In [ ]:
RUN_LR_IMBALANCE_SCREENING = False

if RUN_LR_IMBALANCE_SCREENING:
    lr_result = train_logistic_regression(
        static,
        splits,
        profile='screening',
        progress_callback=print,
    )
    lr_artifacts = save_static_training_result(lr_result, artifact_suffix=ARTIFACT_SUFFIX)
    display(lr_result.strategy_metrics)
    print(f'Final LR candidate: {lr_result.best_candidate.name}')
    print(f'Protected LR OOF: {lr_artifacts.oof_path}')
else:
    print('LR screening is gated. Set RUN_LR_IMBALANCE_SCREENING=True when ready.')

In [ ]:
RUN_XGB_IMBALANCE_SCREENING = False

if RUN_XGB_IMBALANCE_SCREENING:
    xgb_result = train_xgboost(
        static,
        splits,
        profile="screening",
        device="cuda",
        progress_callback=print,
    )
    xgb_artifacts = save_static_training_result(xgb_result, artifact_suffix=ARTIFACT_SUFFIX)
    display(xgb_result.strategy_metrics)
    print(f'Final XGBoost candidate: {xgb_result.best_candidate.name}')
    print(f'Protected XGBoost OOF: {xgb_artifacts.oof_path}')

## 5. SVM: grouped calibration, smoke, then tuning

SVM uses three patient-grouped inner folds to fit sigmoid probability calibration within each outer training fold. Preprocessing and resampling are fitted separately inside those folds. `SVC` runs on CPU with `probability=False`; the saved calibrated model exposes `predict_proba()`.

Smoke compares three strategies; tuning compares 12 kernel/C/gamma settings across three strategies. Inner calibration makes these runs expensive, so measure smoke runtime before enabling tuning. Both gates start off.

In [ ]:
svm_smoke_candidates = svm_candidates(profile='smoke')
svm_tuning_candidates = svm_candidates(profile='tuning')
print(f'SVM Smoke candidates: {len(svm_smoke_candidates)}')
print(f'SVM Tuning candidates: {len(svm_tuning_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in svm_tuning_candidates]))

# Keep new calibration runs separate from previous model artifacts/checkpoints.
SVM_ARTIFACT_SUFFIX = 'simple_svm_v1'
SVM_CHECKPOINT_DIR = DATA_PROCESSED / 'checkpoints' / f'svm_{SVM_ARTIFACT_SUFFIX}'

In [ ]:
RUN_SVM_SMOKE = False
svm_smoke_result = None

if RUN_SVM_SMOKE:
    svm_smoke_result = train_svm(
        static, splits, profile='smoke',
        checkpoint_dir=None, progress_callback=print,
    )
    svm_smoke_artifacts = save_static_training_result(
        svm_smoke_result, artifact_suffix=f'{SVM_ARTIFACT_SUFFIX}_smoke'
    )
    display(svm_smoke_result.strategy_metrics)
    print(f'SVM smoke best candidate: {svm_smoke_result.best_candidate.name}')
else:
    print('SVM smoke is gated. Set RUN_SVM_SMOKE=True when ready.')

### Checkpoint recovery

The notebook uses a new directory for this simplified version. Completed outer candidate/folds can be reused after an interruption; an unfinished fold reruns. Set `checkpoint_dir=None` to turn saving off.

A small manifest checks data, splits and model settings. Use a new directory and artifact suffix after changing code or dependencies; these are no longer checked automatically. Run one experiment per directory in the controlled environment.

In [ ]:
RUN_SVM_TUNING = False
svm_tuning_result = None

if RUN_SVM_TUNING:
    svm_tuning_result = train_svm(
        static, splits, profile='tuning',
        checkpoint_dir=SVM_CHECKPOINT_DIR,
        resume=True, progress_callback=print,
    )
    svm_tuning_artifacts = save_static_training_result(
        svm_tuning_result, artifact_suffix=f'{SVM_ARTIFACT_SUFFIX}_tuning'
    )
    display(svm_tuning_result.strategy_metrics)
    print(f'Final SVM candidate: {svm_tuning_result.best_candidate.name}')
else:
    print('SVM tuning is gated. Set RUN_SVM_TUNING=True after checking smoke.')

## 6. PyTorch MLP: grouped early stopping, smoke, then tuning

MLP uses the same static features and outer folds. A patient-grouped inner holdout selects the epoch count by validation AUROC, then a fresh network refits on the full current training partition. Preprocessing, resampling and class weights use only that partition's training rows. The saved model predicts on CPU.

Smoke compares three strategies with at most five epochs. Tuning samples 10 network settings across three strategies (30 candidates). Check smoke runtime before enabling tuning.

In [ ]:
import torch

MLP_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'MLP training device: {MLP_DEVICE}; PyTorch: {torch.__version__}')
mlp_smoke_candidates = mlp_candidates(profile='smoke')
mlp_tuning_candidates = mlp_candidates(profile='tuning')
print(f'MLP Smoke candidates: {len(mlp_smoke_candidates)}')
print(f'MLP Tuning candidates: {len(mlp_tuning_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in mlp_tuning_candidates]))

MLP_ARTIFACT_SUFFIX = 'simple_mlp_v1'
MLP_CHECKPOINT_DIR = DATA_PROCESSED / 'checkpoints' / f'mlp_{MLP_ARTIFACT_SUFFIX}'

In [ ]:
RUN_MLP_SMOKE = False
mlp_smoke_result = None

if RUN_MLP_SMOKE:
    mlp_smoke_result = train_mlp(
        static, splits, profile='smoke', device=MLP_DEVICE,
        checkpoint_dir=None, progress_callback=print,
    )
    mlp_smoke_artifacts = save_static_training_result(
        mlp_smoke_result, artifact_suffix=f'{MLP_ARTIFACT_SUFFIX}_smoke'
    )
    display(mlp_smoke_result.strategy_metrics)
    display(mlp_smoke_result.fold_metrics[[
        'candidate', 'fold', 'selected_epochs', 'selection_epochs',
        'inner_validation_auroc', 'fit_seconds',
    ]])
    print(f'MLP smoke best candidate: {mlp_smoke_result.best_candidate.name}')
else:
    print('MLP smoke is gated. Set RUN_MLP_SMOKE=True when ready.')

### MLP training and recovery

Tuning allows 100 epochs with patience 10, then refits for the selected epoch count. Each candidate/fold can train two networks. Recovery is per completed outer fold. Select CUDA when available, or CPU.

In [ ]:
RUN_MLP_TUNING = False
mlp_tuning_result = None

if RUN_MLP_TUNING:
    mlp_tuning_result = train_mlp(
        static, splits, profile='tuning', device=MLP_DEVICE,
        checkpoint_dir=MLP_CHECKPOINT_DIR,
        resume=True, progress_callback=print,
    )
    mlp_tuning_artifacts = save_static_training_result(
        mlp_tuning_result, artifact_suffix=f'{MLP_ARTIFACT_SUFFIX}_tuning'
    )
    display(mlp_tuning_result.strategy_metrics)
    display(mlp_tuning_result.fold_metrics[[
        'candidate', 'fold', 'selected_epochs', 'selection_epochs',
        'inner_validation_auroc', 'fit_seconds',
    ]])
    print(f'Final MLP candidate: {mlp_tuning_result.best_candidate.name}')
else:
    print('MLP tuning is gated. Set RUN_MLP_TUNING=True after checking smoke.')

## Shared training flow

`train_static_model` validates frozen splits, fits each candidate across development folds, records AUROC/AUPRC/Brier, retains each strategy's best OOF predictions, and refits the overall winner on all development rows. LR, XGBoost, RF, SVM and MLP use this flow; SVM adds grouped calibration and MLP adds grouped early stopping. The hourly LSTM lives in `src/models/lstm.py`.

Internal-test prediction and F1 threshold selection belong to Stage 5. Review only aggregate metrics and counts outside the controlled environment.